# Benchmark: current pipeline vs. archive's legacy pipeline

Runs both pipelines across a folder of PDFs and scores them against the same
auto-labelled ground truth on the same metrics:

- **Current**: `rastervec`'s own `Pipeline.STAGES` chain (layer/color separation
  -> 12-step Vector_Classification -> FAST text detect -> PaddleOCR), via
  `pipeline.run_page_context`.
- **Archive (legacy)**: `archive/raster_parser/main_pipeline_extract.extract`,
  run completely unmodified via `Evaluation/Evaluate/legacy_adapter.py`.

Ground truth for both comes from `auto_label_pdf` -- native-text-derived,
independent of either pipeline under test, so neither has an unfair advantage.

This is a **sanity/regression check**, not a real A/B: the current pipeline is
expected to score at least as well as the archive one. If it doesn't on some
PDF, that's worth a closer look.

In [ ]:
import sys
from pathlib import Path
import tempfile

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd() / "benchmark_vector_classification.ipynb").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pymupdf as fitz
from tqdm import tqdm

from rastervec.Evaluation.Conversion.conversion import convert_page_to_vector_text
from rastervec.Evaluation.Evaluate.evaluate import DEFAULT_IOU_THRESHOLD, evaluate_pipeline
from rastervec.Evaluation.Evaluate.benchmark import aggregate_results, format_report
from rastervec.Evaluation.Evaluate.legacy_adapter import (
    run_archive_pipeline,
    to_cluster_ocr_results,
    to_drawing_vectors,
)
from rastervec.Evaluation.Labelling.auto_label import auto_label_pdf
from rastervec.logging_setup import configure_logging
from rastervec.pipeline import run_page_context
from rastervec.Reader.reader import Reader

configure_logging()

## Parameters

Point `PDF_FOLDER` at a folder of PDFs (e.g. `references/`, gitignored --
not checked into this repo). `PAGES_PER_PDF` caps how many pages of each PDF
get benchmarked (`None` = every page -- can be slow, since both pipelines run
real PaddleOCR, and the archive pipeline also shells out to LibreOffice).

In [ ]:
PDF_FOLDER = PROJECT_ROOT / "references"
PAGES_PER_PDF = 3
IOU_THRESHOLD = DEFAULT_IOU_THRESHOLD

# Archive's raster-fallback stage shells out to LibreOffice (`soffice`) to
# strip native content before its Type-4 OCR/autotrace pass -- set True only
# if LibreOffice is installed and on PATH; otherwise this stays Type-2-only
# (native + fill-vector OCR), which is still a fair comparison since the
# current pipeline being benchmarked has no raster-image stage either.
ENABLE_ARCHIVE_RASTER_PASS = False

In [ ]:
def iter_pdf_pages(folder: Path, max_pages: int | None):
    for pdf_path in sorted(folder.glob("*.pdf")):
        doc = fitz.open(str(pdf_path))
        n_pages = len(doc) if max_pages is None else min(max_pages, len(doc))
        doc.close()
        for page_index in range(n_pages):
            yield str(pdf_path), page_index


pdf_pages = list(iter_pdf_pages(PDF_FOLDER, PAGES_PER_PDF))
print(f"{len(pdf_pages)} (pdf, page) pairs found under {PDF_FOLDER}")

In [ ]:
def run_current(pdf_path: str, page_index: int, iou_threshold: float):
    """Ground truth (pipeline-independent, on the ORIGINAL native PDF) ->
    Conversion (native text -> vector-drawn text, matching benchmark.py's
    own run_one_page) -> real current-pipeline run (OCR included) ->
    evaluate_pipeline.

    Conversion matters here: the current pipeline's Vector_Classification +
    OCR chain only ever scores *vector-path-drawn* text (CAD text-as-paths),
    never real native font text directly -- on an unconverted native PDF
    there's nothing for it to find at all, so every label would come back
    "not_found" regardless of how good the classifier is. Archive's own
    pipeline doesn't need this since it folds real native words straight
    into its output alongside OCR'd fill-vector text.
    """
    labels = auto_label_pdf(pdf_path, page_index)
    converted_bytes = convert_page_to_vector_text(pdf_path, page_index)

    with tempfile.TemporaryDirectory() as tmp_dir:
        converted_path = str(Path(tmp_dir) / "converted.pdf")
        Path(converted_path).write_bytes(converted_bytes)
        with Reader(converted_path) as reader:
            ctx = run_page_context(reader, 0)

    return evaluate_pipeline(
        labels,
        ctx.cluster_ocr_results or [],
        ctx.drawing_vectors or [],
        rotation_checks=ctx.rotation_checks,
        iou_threshold=iou_threshold,
        clustering=ctx.clustering,
        fast_dropped=ctx.fast_dropped,
        ocr_failed=ctx.ocr_failed,
    )


def run_legacy(pdf_path: str, page_index: int, iou_threshold: float):
    """Same ground truth -> archive's own pipeline, run unmodified on the
    ORIGINAL native PDF via legacy_adapter (no Conversion needed -- archive
    reads native text directly) -> evaluate_pipeline."""
    labels = auto_label_pdf(pdf_path, page_index)
    elements = run_archive_pipeline(
        pdf_path, page_index, enable_raster_pass=ENABLE_ARCHIVE_RASTER_PASS,
    )
    cluster_ocr_results = to_cluster_ocr_results(elements, page_index=page_index)
    drawing_vectors = to_drawing_vectors(elements, page_index=page_index)
    return evaluate_pipeline(
        labels, cluster_ocr_results, drawing_vectors, iou_threshold=iou_threshold,
    )

In [ ]:
current_results = []
legacy_results = []

for pdf_path, page_index in tqdm(pdf_pages, desc="benchmarking"):
    try:
        result = run_current(pdf_path, page_index, IOU_THRESHOLD)
        current_results.append(result)
        print(format_report(f"[current] {pdf_path}", page_index, result))
    except Exception as exc:  # noqa: BLE001 -- keep benchmarking the rest of the folder
        print(f"[current] {pdf_path} page {page_index} failed: {exc}")

    try:
        result = run_legacy(pdf_path, page_index, IOU_THRESHOLD)
        legacy_results.append(result)
        print(format_report(f"[legacy]  {pdf_path}", page_index, result))
    except Exception as exc:  # noqa: BLE001
        print(f"[legacy] {pdf_path} page {page_index} failed: {exc}")

In [ ]:
current_agg = aggregate_results(current_results)
legacy_agg = aggregate_results(legacy_results)

print("Current pipeline (aggregate):")
for key, value in current_agg.items():
    print(f"  {key}: {value}")

print("\nArchive / legacy pipeline (aggregate):")
for key, value in legacy_agg.items():
    print(f"  {key}: {value}")

In [ ]:
import matplotlib.pyplot as plt

_METRICS = (
    "characters_found_pct",
    "character_accuracy",
    "rotation_accuracy",
    "bbox_accuracy",
    "classification_precision",
    "classification_recall",
)

current_vals = [current_agg.get(m, 0.0) for m in _METRICS]
legacy_vals = [legacy_agg.get(m, 0.0) for m in _METRICS]

x = range(len(_METRICS))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - width / 2 for i in x], current_vals, width, label="current")
ax.bar([i + width / 2 for i in x], legacy_vals, width, label="archive (legacy)")
ax.set_xticks(list(x))
ax.set_xticklabels(_METRICS, rotation=30, ha="right")
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("Current pipeline vs. archive pipeline")
plt.tight_layout()
plt.show()

## Reading the results

The current pipeline is expected to match or beat the archive pipeline on
every metric -- it's the newer, cluster-based approach the archive pipeline
was replaced by. If a specific PDF/page shows the archive pipeline ahead on
some metric, that's a concrete regression worth investigating (check
`miss_reason_counts` in the per-page `format_report` output above for where
the current pipeline lost that text).